# Lab 3 — Spark Declarative Pipelines concepts (Course 3)
Lessons 3.1–3.5 · exercises `3a`–`3d` · schema `lab3`

☁ In Databricks you **declare** tables in SQL and SDP runs them:
```sql
CREATE OR REFRESH STREAMING TABLE orders_bronze AS
  SELECT *, current_timestamp() AS processing_time, _metadata.file_name AS source_file
  FROM STREAM read_files('${source}/orders', format => 'JSON');
CREATE OR REFRESH STREAMING TABLE orders_silver AS
  SELECT order_id, timestamp(order_timestamp) AS order_timestamp, customer_id, notifications
  FROM STREAM orders_bronze;
CREATE OR REFRESH MATERIALIZED VIEW gold_orders_by_date AS
  SELECT date(order_timestamp) AS order_date, count(*) AS total_daily_orders
  FROM orders_silver GROUP BY date(order_timestamp);
```
🖥 Locally we build the **same three datasets by hand** with Structured Streaming. You will see exactly what SDP
automates for you: checkpoints, dependency order, incremental processing, recomputing the MV.

**How this lab works**
- Run cells top to bottom (`Shift+Enter`). After a kernel restart, run again from the top.
- Cells marked `# TODO` contain `<FILL-IN>`: replace it with your code, then run the `check(...)` cell.
- Stuck? Ask Claude in the terminal: *"hint for 1c"*. Solutions are in `notebooks/solutions/` (try first!).
- 🖥 **Local** means open-source Spark + Delta on your Mac. ☁ **Databricks** shows the real syntax for the exam.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))          # lets the notebook import labutils
from labutils import get_spark, path, reset_lab, show, checkpoint, table_path, landing
from labutils.check import check
from pyspark.sql import functions as F
spark = get_spark()
reset_lab("lab3")
for v in ("orders", "status", "customers"):
    landing.reset(v)
orders_schema = "order_id BIGINT, order_timestamp STRING, customer_id BIGINT, notifications STRING"
status_schema = "order_id BIGINT, order_status STRING, status_timestamp STRING"

## 3a · Streaming tables + materialized view (lesson 3.1)
- **Streaming table** = append-only, each input processed **once** (checkpoint). Reading from a ST uses `STREAM`.
- **Materialized view** = result is **recomputed** to match the current data (SDP refreshes incrementally when it can).
- **Task:** complete silver (read bronze as a stream) and gold (daily counts).

In [ ]:
def refresh_orders_bronze():
    (spark.readStream.format("json").schema(orders_schema).load(landing.volume("orders"))
          .select("*", F.current_timestamp().alias("processing_time"),
                  F.col("_metadata.file_name").alias("source_file"))
     .writeStream.option("checkpointLocation", checkpoint("lab3", "orders_bronze"))
          .trigger(availableNow=True).toTable("lab3.orders_bronze").awaitTermination())

def refresh_orders_silver():
    (spark.readStream.table("lab3.orders_bronze")
          .select("order_id", F.to_timestamp("order_timestamp").alias("order_timestamp"),
                  "customer_id", "notifications")
     .writeStream.option("checkpointLocation", checkpoint("lab3", "orders_silver"))
          .trigger(availableNow=True).toTable("lab3.orders_silver").awaitTermination())

def refresh_gold():
    spark.sql("""CREATE OR REPLACE TABLE lab3.gold_orders_by_date AS
                 SELECT date(order_timestamp) AS order_date, count(*) AS total_daily_orders
                 FROM lab3.orders_silver
                 GROUP BY date(order_timestamp)""")

def run_pipeline():
    refresh_orders_bronze(); refresh_orders_silver(); refresh_gold()

In [ ]:
def counts(label):
    print(label, {t: spark.table(f"lab3.{t}").count() for t in ("orders_bronze", "orders_silver", "gold_orders_by_date")})

landing.land("orders", 2); run_pipeline(); counts("pipeline update 1:")
landing.land("orders", 2); run_pipeline(); counts("pipeline update 2:")
show(spark.table("lab3.gold_orders_by_date").orderBy("order_date"))
check("3a")

## 3b · Data quality expectations (lesson 3.3)
`CONSTRAINT name EXPECT (condition) [ON VIOLATION DROP ROW | FAIL UPDATE]`
| action | bad rows | pipeline |
|---|---|---|
| (none) = WARN | kept | continues, metrics logged |
| `DROP ROW` | removed | continues, metrics logged |
| `FAIL UPDATE` | — | update stops |

🖥 Local helper: `expect(df, [(name, condition, "warn" | "drop" | "fail")])` prints event-log-style metrics.
**Task:** `valid_notifications` (`notifications IN ('Y','N')`) should only **warn**; `valid_id` (`customer_id IS NOT NULL`) should **drop**.

In [ ]:
from labutils.sdp import expect, ExpectationFailed
silver_dq, metrics = expect(spark.table("lab3.orders_silver"), [
    ("valid_notifications", "notifications IN ('Y','N')", "warn"),
    ("valid_id",            "customer_id IS NOT NULL",    "drop"),
], dataset="orders_silver_dq")
silver_dq.write.mode("overwrite").saveAsTable("lab3.orders_silver_dq")

In [ ]:
check("3b")
# What does FAIL UPDATE do?
try:
    expect(spark.table("lab3.orders_silver"), [("valid_id", "customer_id IS NOT NULL", "fail")], dataset="strict")
except ExpectationFailed as e:
    print("PIPELINE UPDATE FAILED:", e)

## 3c · Joining two streaming tables in a materialized view (lesson 3.4)
To join **all rows** of two streaming tables every update, SDP uses a **materialized view**
(stream–stream joins need watermarks and are out of course scope).
**Task:** build the status pipeline, then join orders and statuses in `full_order_info_gold`.

In [ ]:
def refresh_status():
    (spark.readStream.format("json").schema(status_schema).load(landing.volume("status"))
     .writeStream.option("checkpointLocation", checkpoint("lab3", "status_bronze"))
          .trigger(availableNow=True).toTable("lab3.status_bronze").awaitTermination())
    (spark.readStream.table("lab3.status_bronze")
          .select("order_id", "order_status", F.to_timestamp("status_timestamp").alias("status_timestamp"))
     .writeStream.option("checkpointLocation", checkpoint("lab3", "status_silver"))
          .trigger(availableNow=True).toTable("lab3.status_silver").awaitTermination())

landing.land("status", 4); refresh_status()

In [ ]:
spark.sql("""
CREATE OR REPLACE TABLE lab3.full_order_info_gold AS
SELECT o.order_id, o.order_timestamp, o.customer_id, s.order_status, s.status_timestamp
FROM lab3.orders_silver o
INNER JOIN lab3.status_silver s ON o.order_id = s.order_id
""")

In [ ]:
show(spark.sql("SELECT order_status, count(*) n FROM lab3.full_order_info_gold GROUP BY 1"))
check("3c")

## 3d · Change Data Capture with SCD Type 1 (lesson 3.5)
☁ Databricks:
```sql
CREATE OR REFRESH STREAMING TABLE customers_silver;
CREATE FLOW scd_type_1_flow AS AUTO CDC INTO customers_silver
FROM STREAM customers_bronze
KEYS (customer_id)
APPLY AS DELETE WHEN operation = 'DELETE'
SEQUENCE BY process_date
COLUMNS * EXCEPT (operation)
STORED AS SCD TYPE 1;
```
🖥 Locally we rebuild AUTO CDC by hand: `foreachBatch` + `MERGE`.
The 2nd batch has **two updates for Peter that arrive out of order** (05-20 first, then 05-15). `SEQUENCE BY` makes the
**latest process_date win**. **Task:** keep only the latest change per key, and apply deletes.

In [ ]:
from pyspark.sql.window import Window
cdc_schema = "customer_id BIGINT, name STRING, address STRING, operation STRING, process_date STRING"
spark.sql("CREATE OR REPLACE TABLE lab3.customers_silver (customer_id BIGINT, name STRING, address STRING, process_date STRING)")

In [ ]:
def apply_changes(batch_df, batch_id):
    w = Window.partitionBy("customer_id").orderBy(F.col("process_date").desc())
    latest = batch_df.withColumn("rn", F.row_number().over(w)).filter("rn = 1").drop("rn")
    latest.createOrReplaceTempView("changes")
    latest.sparkSession.sql("""
      MERGE INTO lab3.customers_silver t USING changes s ON t.customer_id = s.customer_id
      WHEN MATCHED AND s.operation = 'DELETE' THEN DELETE
      WHEN MATCHED AND s.process_date >= t.process_date THEN
           UPDATE SET t.name = s.name, t.address = s.address, t.process_date = s.process_date
      WHEN NOT MATCHED AND s.operation != 'DELETE' THEN
           INSERT (customer_id, name, address, process_date) VALUES (s.customer_id, s.name, s.address, s.process_date)
    """)

def refresh_customers():
    (spark.readStream.format("json").schema(cdc_schema).load(landing.volume("customers"))
     .writeStream.foreachBatch(apply_changes)
          .option("checkpointLocation", checkpoint("lab3", "customers_cdc"))
          .trigger(availableNow=True).start().awaitTermination())

In [ ]:
landing.land("customers", 1); refresh_customers()
print("after batch 1:"); show(spark.table("lab3.customers_silver").orderBy("customer_id"))
landing.land("customers", 1); refresh_customers()
print("after batch 2 (update, out-of-order update, delete, insert):"); show(spark.table("lab3.customers_silver").orderBy("customer_id"))
check("3d")

## Next
- In Databricks Free Edition, open course 3's `2 Demo` and build the real pipeline in the **Lakeflow Pipelines Editor**:
  see the pipeline graph, dry run, full refresh, and the event log.
- Ask Claude: `/quiz sdp 5` or `/explain streaming table vs materialized view`.